# 01 | Source Data Inspection

**Project:** Dubai Short-Term Rental Intelligence  |  **Source:** AirROI Dubai (Kaggle download)  |  **Reporting coverage:** March 2021–February 2026

**Business question:** What can be reliably measured from the available files? This assessment establishes the observation grain, source overlap, history, currency, and feasible analytical scope before transformations.

**Interpretation:** Figures describe the supplied AirROI sample. Occupancy and revenue are provider estimates, not audited bookings or market-wide totals. The source is independent of Airbnb.

## Data provenance and reproducibility

**Source:** AirROI Dubai dataset distributed via Kaggle (third-party provider; not an official Airbnb dataset). All four files are inspected. CSV history and rates Parquet are reconciled; listing Parquet supplies broader snapshot metadata and nested native booked rates.

**Execution environment:** Python 3 with dependencies in `requirements.txt`. Source files are read from `data/raw/` in the project root and are not redistributed in this GitHub release while usage rights remain unverified. Any derived audits are written outside `data/raw/` to preserve provenance. The steps and source attribution are documented in the project README.

**Snapshot control (independent audit, 10 October 2026):** Fixed figures in this narrative refer to the four original files identified by SHA-256 in `reports/source_audit.json`. Executable tables derive values at runtime. Review the narrative if replacing the source snapshot. Raw data and listing-level outputs remain private; conclusions describe this sample and provider estimates.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

CWD = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [CWD, CWD.parent, CWD.parent.parent]
                     if (p / 'README.md').exists() and (p / 'data' / 'raw').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run Jupyter from the project root or notebooks/ directory.')
RAW = PROJECT_ROOT / 'data' / 'raw'
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
REPORTS = PROJECT_ROOT / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 18)
pd.set_option('display.max_rows', 20)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')
print('Project:', PROJECT_ROOT.name)

## File inventory and source structure

`past_rates.csv` is the historical listing-month dataset. `listings_monthly.csv` contains additional listing attributes for a subset of overlapping listing-months. IDs are loaded as text to preserve precision.

In [2]:
files = ['past_rates.csv', 'listings_monthly.csv', 'listings.parquet', 'past_rates.parquet']
file_inventory = pd.DataFrame([{'file': f, 'present': (RAW/f).is_file(),
                                'size_mb': round((RAW/f).stat().st_size/(1024**2), 2) if (RAW/f).is_file() else None}
                               for f in files])
display(file_inventory)
required = ['past_rates.csv', 'listings_monthly.csv']
missing = [f for f in required if not (RAW/f).exists()]
if missing:
    raise FileNotFoundError(f'Missing source files: {missing}. Restore the original AirROI download to data/raw/.')

history = pd.read_csv(RAW/'past_rates.csv', dtype={'listing_id':'string'}, low_memory=False)
enriched = pd.read_csv(RAW/'listings_monthly.csv', dtype={'listing_id':'string','host_id':'string'}, low_memory=False)
for data in (history, enriched):
    data['month_date'] = pd.to_datetime(data['month_date'], errors='raise')
print('History:',history.shape,'Enriched:',enriched.shape)

## Granularity, uniqueness and relationship

One row represents a **listing × calendar month**, not a guest stay or transaction. The enriched CSV is checked for overlap before modeling; its rows are not additive to historical observations.

In [3]:
KEY=['listing_id','month_date']
source_summary=pd.DataFrame([{
    'dataset':name, 'rows':len(frame),'columns':frame.shape[1],
    'unique_listings':frame.listing_id.nunique(),
    'months':frame.month_date.nunique(),
    'first_month':frame.month_date.min().strftime('%Y-%m'),
    'last_month':frame.month_date.max().strftime('%Y-%m'),
    'duplicate_keys':int(frame.duplicated(KEY).sum()),
    'missing_ids':int(frame.listing_id.isna().sum())
} for name,frame in [('past_rates.csv',history),('listings_monthly.csv',enriched)]])
display(source_summary)
assert not history.duplicated(KEY).any() and not enriched.duplicated(KEY).any()
matched=enriched[KEY].merge(history[KEY],on=KEY,how='left',indicator=True,validate='one_to_one')
display(matched['_merge'].value_counts().rename_axis('join_result').reset_index(name='records'))
assert (matched['_merge']=='both').all(), 'The enriched source contains keys absent from the primary file.'

## Field coverage and currency

Missingness is calculated without replacing missing values. The `native_*` monetary measures use the source's recorded AED currency; similarly named non-native fields are **not** assumed to be AED.

In [4]:
def profile(frame):
    return (pd.DataFrame({'field':frame.columns,
                          'dtype':[str(frame[col].dtype) for col in frame],
                          'missing_pct':[round(frame[col].isna().mean()*100,2) for col in frame]})
            .sort_values(['missing_pct','field'],ascending=[False,True]))
display(profile(history).head(12))
display(profile(enriched).head(12))
print('Currencies:',history.currency.value_counts(dropna=False).to_dict())
print('Max listing ID length:',int(history.listing_id.str.len().max()))
print('Reference rate fields:',[x for x in history if 'rate' in x or 'revenue' in x])

## Historical sample coverage

A change in the number of records collected is not equivalent to a change in the Dubai rental market. Reporting will include sample coverage alongside historical trends.

In [5]:
import matplotlib.pyplot as plt
coverage=(history.groupby('month_date').agg(observed=('listing_id','nunique'), active=('active','sum'))
          .sort_index().reset_index())
coverage['observed_mom_change_pct']=coverage.observed.pct_change(fill_method=None)*100
display(coverage.tail(5))
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(coverage.month_date,coverage.observed,label='Observed listings')
ax.plot(coverage.month_date,coverage.active,label='Active listings')
ax.set(title='AirROI sample coverage by reporting month',xlabel='Month',ylabel='Listings')
ax.legend(); ax.grid(alpha=.15);fig.tight_layout()
figdir=REPORTS/'figures';figdir.mkdir(exist_ok=True)
fig.savefig(figdir/'01_source_coverage.png',dpi=140)
plt.show()
print('Largest absolute changes:')
display(coverage.loc[coverage.observed_mom_change_pct.abs().nlargest(5).index,
                 ['month_date','observed','observed_mom_change_pct']].sort_values('month_date'))

## Latest reporting month and benchmark eligibility and Parquet availability

Parquet inspection is required. The full comparison checks all historical keys and shared values, plus nested monthly records; snapshot metadata is not assumed historically valid.

In [6]:
latest=history.month_date.max()
recent=history[history.month_date==latest]
active=recent[recent.active]
summary={'latest_month':latest.strftime('%Y-%m'),
         'observed_listings':int(recent.listing_id.nunique()),
         'active_listings':int(active.listing_id.nunique()),
         'missing_neighborhood_pct':round(recent.neighborhood.isna().mean()*100,2),
         'missing_bedrooms_pct':round(recent.bedrooms.isna().mean()*100,2),
         'zero_estimated_occupancy_active_pct':round((active.occupancy==0).mean()*100,2),
         'active_with_initial_peer_fields':int((active[['latitude','longitude','room_type','bedrooms','native_avg_daily_rate']].notna().all(axis=1)
                                          & active.native_avg_daily_rate.gt(0)
                                          & active.latitude.between(-90,90)
                                          & active.longitude.between(-180,180)).sum())}
display(pd.DataFrame([summary]).T.rename(columns={0:'result'}))
import runpy
source_audit = runpy.run_path(str(PROJECT_ROOT/'src/source_audit.py'))['run']()
display(pd.DataFrame(source_audit['sources']).T)
print('Shared monthly disagreements:', sum(source_audit['nested_monthly_disagreements'].values()))
print('Additional native booked-rate observations:', source_audit['native_booked_rate_nonnull'])


## Findings and analytical decisions

**Inspection outcome:** The supplied files support a historical, property-level market intelligence analysis, subject to sampling and modeled-metric limitations.

| Evidence from the supplied dataset | Verified observation | Analytical consequence |
|---|---:|---|
| Historical property-month records | **99,745** | Use `listing_id` + `month_date` as the observation key. |
| Distinct historical properties | **4,364** | A property may occur in multiple reporting months. |
| Enriched property-month rows | **22,629** | These overlap the main history and **must not** be added to its row count. |
| Observed reporting range | **March 2021–February 2026** (60 months) | Historical comparisons require explicit dates and source-coverage checks. |
| Listings observed / active in February 2026 | **2,471 / 1,955** | Publish the reporting denominator with all active-listing KPIs. |
| February 2026 records missing a neighborhood label | **77.7%** | Prefer available coordinates and verified distance-based cohorts to unsupported neighborhood rankings. |
| Active February 2026 listings with zero provider-estimated occupancy | **47.0%** | Audit source estimates before interpreting utilization or modeled revenue. |
| Active February 2026 records with core preliminary peer-matching inputs | **1,717** (computed above) | This is input completeness only; competitor matching and minimum peer counts are checked downstream. |

These figures refer **only to the supplied AirROI sample**, not all Dubai short-term rentals. The AirROI performance fields are estimates; there is no verified stay, booking, profit, or actual occupancy ground truth in this inspection. The rates Parquet duplicates CSV history. The listing Parquet contains all 4,364 properties and 81 fields; its nested history matches CSV measures and adds native booked-rate estimates. Undated metadata enriches the listing dimension only.

**Decision for the analytical pipeline:** Preserve the raw inputs, retain long-format listing-month history, treat the monthly enrichment as supplemental attributes, and keep coverage/missingness indicators in every downstream report. Native monetary metrics are in **AED for this supplied data**; do not mix them with other currency fields.

**Next analytical stage:** `02_data_quality_assessment.ipynb` evaluates range checks, enrichment agreement, missingness, coverage changes, and the reliability of benchmark inputs before data preparation begins.